# 01 — Financial Statement Analysis with Claude

This notebook walks through analysing a 10-K excerpt using the `financial_analysis` prompt template.

**Prerequisites:**
```bash
pip install anthropic pandas
export ANTHROPIC_API_KEY="sk-ant-..."
```

In [ ]:
import sys, json
sys.path.insert(0, '../scripts')
from utils import load_template, fill_template, extract_user_prompt, extract_system_prompt, call_claude, extract_json

## 1. Load the prompt template

In [ ]:
template = load_template('financial_analysis')
print(template[:500])

## 2. Prepare the report

Paste or load a 10-K excerpt. Here we use a sample.

In [ ]:
# Replace this with a real filing excerpt
sample_report = """
ACME Corp — Annual Report FY2025

Revenue: $2.4B (up 18% YoY)
Gross margin: 62%
Operating margin: 28%
Net income: $480M
Total assets: $8.1B
Total debt: $1.2B
Cash and equivalents: $3.4B
Free cash flow: $720M

Segment breakdown:
- Payments: $1.5B (63% of revenue, +22% YoY)
- Lending: $600M (25% of revenue, +12% YoY)
- Data Services: $300M (12% of revenue, +15% YoY)

Key risks: Regulatory changes in EU markets, increasing competition
from neobanks, concentration risk in top 10 enterprise clients.
"""

print(f'Report length: {len(sample_report):,} chars')

## 3. Fill the template and call Claude

In [ ]:
system_prompt = extract_system_prompt(template)
user_prompt = extract_user_prompt(template)
user_prompt = fill_template(user_prompt, {
    'REPORT': sample_report,
    'COMPANY_NAME': 'ACME Corp',
    'FISCAL_PERIOD': 'FY2025',
})

print('System prompt:', system_prompt[:200])
print('\nUser prompt (first 300 chars):', user_prompt[:300])

In [ ]:
raw_response = call_claude(user_prompt, system_prompt=system_prompt)
result = extract_json(raw_response)
print(json.dumps(result, indent=2))

## 4. Work with the structured output

Since the result is JSON, you can pipe it into dashboards, databases, or downstream analysis.

In [ ]:
import pandas as pd

if 'revenue' in result and 'segments' in result['revenue']:
    df = pd.DataFrame(result['revenue']['segments'])
    print(df.to_string(index=False))
else:
    print('No segment data found — check the report excerpt.')

---

**Next:** Try `02_forecasting.ipynb` to generate revenue projections from this data.